# Quick Start

This notebook provides inference pipeline for pretrained FSM-LVSM model on a **single scene**.

This should run on a single A6000 machine and finish in a second.

It is intended for smoke-testing your setup, debugging configs, etc.

## Prerequisites

Run this cell first. If any import fails, ensure your virtual environment is activated and dependencies are installed.

In [ ]:
import os
import shutil
import yaml
import json
from pathlib import Path
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download

import math
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from lpips import LPIPS

These import from the local utility files. Make sure you are running this notebook from the **project root** (where `fsm/` and `utils_train.py` live), otherwise these will fail with `ModuleNotFoundError`.

In [ ]:
from fsm.data.dataset_fromvid import NVSVideoDataset
from fsm.model.metrics import (
    compute_ssim_from_imgs, 
    save_random_samples_with_psnr
)
from utils_train import (
    get_class_by_name,
    get_ewc_config,
    discover_fast_blocks,
    init_ewc_buffers,
    remap_ewc_buffers,
    build_info
)

`every(n, step)` returns `True` when `step` is a multiple of `n`, or within the first `also_first` steps, used to gate logging, visualization, and checkpointing.

In [ ]:
def every(n, step, also_first=10):
    return (step % n == 0) or (step <= also_first)

## Inference Configurations

Defaults to CUDA if available, CPU otherwise. Training on CPU is possible but very slow, a GPU is strongly recommended.

In [ ]:
# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42
torch.manual_seed(seed)

We will use the provided toy config and experiment name throughout this notebook. Edit these if you want to point to a different setup.

In [ ]:
# ── Hardcoded config ────────────────────────
EXP_NAME      = "fsm_inference"
CONFIG_PATH   = "fsm/configs/inference/fsm_lvsm_inference.yaml"
config = OmegaConf.load(CONFIG_PATH)

For the pretrained model, we use its config:

| Parameter | Value |
|---|---|
| Patch size | 8 |
| Dimension | 768 |
| Layers | 24 |
| Image size | 256 × 256 |
| Max frames | 256 |

In [ ]:
display(yaml.safe_load(OmegaConf.to_yaml(config.model)))

Instantiates the model class specified in the config.
```yaml
model:
  class_name: fsm.model.model_4dlvsm.FSM4DLVSM
```

On first run, this will trigger an automatic download of the LPIPS network weights, expect a short delay.

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
FSM   = get_class_by_name(getattr(config.model, "class_name"))
model = FSM(config).to(device)

Inspect the inference configs.

| Parameter | Value | Description |
|---|---|---|
| `bs_per_gpu` | 1 | Batch size |
| `num_input_views` | 32 | Context views fed to the model |
| `num_target_views` | 8 | Views the model is asked to render |

In [ ]:
display(yaml.safe_load(OmegaConf.to_yaml(config.training)))

## Weight Downloading

Downloads the pretrained LVSM checkpoint from HuggingFace Hub (cached after first download) and copies it to `static/weights/`. 

The checkpoint is ~1.8GB. Download time depends on your connection, subsequent runs will use the local cache instantly.

In [ ]:
repo_id = "marstin/fast-spatial-mem"
local_path = "static/weights"
path_in_repo = "lvsm_checkpoints/fsm_4dlvsm_patch8_res256.pth"

# Download (cached under ~/.cache/huggingface/hub)
cached_path = hf_hub_download(
    repo_id=repo_id,
    filename=path_in_repo,
    repo_type="model"
)

# Copy to your desired local folder
os.makedirs(os.path.dirname(local_path), exist_ok=True)
target_path = os.path.join(local_path, os.path.basename(path_in_repo))
shutil.copy(cached_path, target_path)

print(f"File copied to: {target_path}")

Load the pretrained checkpoint.

In [ ]:
# ── Checkpoint restore ────────────────────────────────────────────────────────
now_iters   = 0
output_dir  = f"outputs/{EXP_NAME}"
os.makedirs(output_dir, exist_ok=True)
ewc_buffers = None
load_ckpt = "static/weights/fsm_4dlvsm_patch8_res256.pth"

if os.path.isdir(output_dir):
        
    checkpoint = torch.load(load_ckpt, map_location="cpu", weights_only=False)
    print(f"Loading checkpoint from {load_ckpt}...")

    msg = model.load_state_dict(checkpoint["model"], strict=True)
    print('Loading msg', msg)

    now_iters = checkpoint.get("now_iters", 0)
    if "ewc_buffers" in checkpoint:
        ewc_buffers = {}
        for k, v in checkpoint["ewc_buffers"].items():
            ewc_buffers[k] = {kk: vv.to(device=device) for kk, vv in v.items()}
        print("Restored EWC buffers from checkpoint.")

Remap EWC buffer keys to strip DDP (`module.`) and activation checkpointing (`_checkpoint_wrapped_module`) prefixes added during distributed training... these are not present in the single-node eval model.

In [ ]:
ewc_buffers = remap_ewc_buffers(checkpoint["ewc_buffers"])

## Dataset

We use a single example scene from Stereo4D for evaluation.

In [ ]:
display(yaml.safe_load(OmegaConf.to_yaml(config.dataset)))

A lightweight wrapper around `NVSVideoDataset` that loads a single scene once and repeats it. The scene is cached in memory after the first load, subsequent iterations have zero I/O overhead. `toy_length` controls how many samples constitute one epoch. With `toy_length=1`.

In [ ]:
class ToyRepeatDataset(Dataset):
    """
    Loads ONE scene from NVSVideoDataset and repeats it `length` times.
    Useful for quick eval without a full dataset.
    """

    def __init__(self, single_scene_dataset: NVSVideoDataset, length: int = 1000):
        self._ds     = single_scene_dataset
        self._length = length
        self._cache  = None   # lazy: load once on first access

    def __len__(self):
        return self._length

    def __getitem__(self, index):
        if self._cache is None:
            self._cache = self._ds[0]
        return self._cache


def get_dataset(
    num_views,
    img_size,
    num_input_views,
    max_frame_time  = 128,
    scene_pose_normalize = True,
    window_size     = 128,
    sorted_indices  = True,
    toy_data_path   = "static/datasets/example/example.json",
    toy_length      = 1000,
):
    """
    Returns a ToyRepeatDataset that yields the same single scene repeatedly,
    and None for the sampler (DataLoader will use shuffle=True instead).
    """
    base_dataset = NVSVideoDataset(
        data_path           = toy_data_path,
        num_views           = num_views,
        image_size          = img_size,
        sorted_indices      = sorted_indices,
        scene_pose_normalize= scene_pose_normalize,
        max_frame_time      = max_frame_time,
        num_input_views     = num_input_views,
        window_size         = window_size,
    )

    dataset = ToyRepeatDataset(base_dataset, length=toy_length)
    sampler = None   # single-node: DataLoader handles shuffling

    print(f"[ToyDataset] Loaded 1 scene from: {toy_data_path}")
    print(f"[ToyDataset] Repeating it {toy_length} times per epoch.")

    return dataset, sampler

Reads view counts and resolution from config. For this toy run: **40 total views** (32 input + 8 target) at **256 × 256** resolution.

In [ ]:
# ── Dataset & dataloader ──────────────────────────────────────────────────────
num_all_views    = config.training.num_all_views
num_input_views  = config.training.num_input_views
num_target_views = config.training.num_target_views

resolution = config.model.image_tokenizer.image_size
img_size   = (resolution, resolution) if isinstance(resolution, int) else tuple(resolution)

Inspect the toy data path, you should see `['static/datasets/example/fZcNop_Aenw_88573338.jsonl']`.

In [ ]:
toy_data_path = "static/datasets/example/example.json"
with open(toy_data_path) as f:
    scene_list = json.load(f)
print(scene_list)

Inspect this scene, you should see:
```bash
dict_keys(['scene_name', 'video_path', 'num_video_frames', 'frames'])
fZcNop_Aenw_88573338
static/datasets/example/fZcNop_Aenw_88573338-left_rectified.mp4
199
dict_keys(['frame_idx', 'w2c', 'fx', 'fy', 'cx', 'cy', 'h', 'w'])
```

In [ ]:
with open(scene_list[0]) as f:
    scene_info = json.load(f)
print(scene_info.keys())
print(scene_info['scene_name'])
print(scene_info['video_path'])
print(scene_info['num_video_frames'])
print(scene_info['frames'][0].keys())

Preview the video.

In [ ]:
from IPython.display import Video
Video(scene_info['video_path'], embed=True, width=640)

Instantiate the dataset and dataloader. Note a few notebook-friendly overrides from the original inference script:

| Parameter | Notebook value | Reason |
|---|---|---|
| `num_workers` | 0 | Avoids multiprocessing issues in Jupyter |
| `persistent_workers` | False | Required when `num_workers=0` |
| `prefetch_factor` | None | Required when `num_workers=0` |
| `drop_last` | False | Inference, keep all batches |

In [ ]:
dataset, sampler = get_dataset(
    num_views=num_all_views,
    img_size=img_size,
    num_input_views=num_input_views,
    max_frame_time=config.model.get("max_frames", 256),
    scene_pose_normalize=config.dataset.get("scene_pose_normalize", True),
    window_size=config.dataset.get("window_size", 256),
    sorted_indices=config.dataset.get("sort_by_time", True),
    toy_data_path=toy_data_path,
    toy_length=1
)

dataloader_seed_generator = torch.Generator()
dataloader_seed_generator.manual_seed(seed)

dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=(sampler is None),
    sampler=sampler,
    num_workers=0,
    persistent_workers=False,
    pin_memory=config.training.get("pin_memory", True),
    drop_last=False,
    prefetch_factor=None,
    generator=dataloader_seed_generator
)

## Model Inference

Initializes Elastic Weight Consolidation (EWC) buffers for the model's fast-weight blocks (LaCT layers). 

These buffers track parameter importance to regularize fast-weight updates during inference.

In [ ]:
# ── EWC config ────────────────────────────────────────────────────────────────
ewc_config      = get_ewc_config(config)
ewc_enable      = ewc_config["enable"]
ewc_lambda_prox = ewc_config["lambda_prox"]
ewc_alpha       = ewc_config["alpha"]
ewc_mode        = ewc_config["mode"]
ewc_anchor      = ewc_config["anchor_policy"]
ewc_anchor_beta = ewc_config["anchor_beta"]
ewc_loss_weight = ewc_config["lambda_train"]

fast_blocks = discover_fast_blocks(model)
if ewc_enable and (ewc_buffers is None):
    print()
    ewc_buffers = init_ewc_buffers(fast_blocks, device=device)

anchor_mode = 1 if "streaming" in ewc_anchor else 0
if "ema" in ewc_anchor:
    anchor_mode *= 2
for _, blk in fast_blocks:
    setattr(blk, "anchor_mode", anchor_mode)
    if "ema" in ewc_anchor:
        setattr(blk, "anchor_beta", ewc_anchor_beta)

fsm_fast_blocks = discover_fast_blocks(model)

Run the inference loop.

In [ ]:
lpips_loss_module = LPIPS(net="vgg").to(device).eval()

cum_total_imgs = 0
cum_sum_psnr   = 0.0
cum_sum_lpips  = 0.0
cum_sum_ssim   = 0.0

print(f"Evaluating {len(dataloader)} batches...")

model.eval()
for data_dict in dataloader:
    data_dict        = {k: v.to(device) for k, v in data_dict.items() if isinstance(v, torch.Tensor)}
    input_data_dict  = {k: v[:, :num_input_views]  for k, v in data_dict.items()}
    target_data_dict = {k: v[:, -num_target_views:] for k, v in data_dict.items()}

    batch_size    = next(iter(data_dict.values())).shape[0]
    fsm_info_dict = build_info(
        fsm_fast_blocks, ewc_buffers, batch_size=batch_size,
        ewc_enable=ewc_enable, lambda_prox=ewc_lambda_prox, alpha=ewc_alpha, mode=ewc_mode
    )

    with torch.no_grad(), torch.autocast(dtype=torch.bfloat16, device_type="cuda", enabled=(device.type == "cuda")):
        rendering, _ = model(input_data_dict, target_data_dict, info=fsm_info_dict, skip_loss=True)
        target = target_data_dict["image"]

        pred = rendering.flatten(0, 1) if rendering.ndim == 5 else rendering
        tgt  = target.flatten(0, 1)    if target.ndim    == 5 else target

        # PSNR
        mse_per_image = F.mse_loss(pred, tgt, reduction='none').mean(dim=(1, 2, 3))
        psnr_per_image = -10.0 * torch.log10(mse_per_image.clamp_min(1e-12))

        # SSIM
        ssim = compute_ssim_from_imgs(rendering, target)

        # LPIPS
        lpips = lpips_loss_module(pred, tgt, normalize=True).mean()

    n_imgs          = pred.shape[0]
    cum_total_imgs += n_imgs
    cum_sum_psnr   += float(psnr_per_image.sum().item())
    cum_sum_lpips  += float(lpips.item()) * n_imgs
    cum_sum_ssim   += float(ssim) * n_imgs

    print(
        f"PSNR {psnr_per_image.mean().item():.2f} | "
        f"LPIPS {lpips.item():.4f} | "
        f"SSIM {ssim:.4f}"
    )

    try:
        save_random_samples_with_psnr(
            rendering=rendering,
            target=target,
            now_iters=0,
            output_dir=output_dir + "_eval",
            sample_n=3,
            wandb_run=None,
            title_prefix="render_vs_gt_with_psnr"
        )
    except Exception as e:
        print(f"[warn] sample logging failed: {e}")
    now_iters += 1

# Final summary
if cum_total_imgs > 0:
    print(
        f"\n[EVAL FINAL] "
        f"avg PSNR {cum_sum_psnr/cum_total_imgs:.3f} | "
        f"avg LPIPS {cum_sum_lpips/cum_total_imgs:.4f} | "
        f"avg SSIM {cum_sum_ssim/cum_total_imgs:.4f} "
        f"over {cum_total_imgs} images"
    )

## Expected Results

The model should be cleanly overfitting the single scene:
```
[EVAL FINAL] avg PSNR 33.996 | avg LPIPS 0.0461 | avg SSIM 0.9384 over 8 images
```

Checkpoints and visualizations are saved to `outputs/fsm_inference_eval/`:
```
outputs/
├── fsm_inference
├── fsm_inference_eval
│   └── 0
│       └── render_vs_gt_with_psnr.png
```

Reconstruction quality should look like:
![quickstart_inference](static/imgs/quickstart_inference.png)